# Souty sign model: two hands + motion (landmark sequences → temporal classifier)

This notebook replaces the MediaPipe Model Maker pipeline. That pipeline had two hard limits, both confirmed in Model Maker's own [`dataset.py`](https://github.com/google-ai-edge/mediapipe/blob/master/mediapipe/model_maker/python/vision/gesture_recognizer/dataset.py):

1. **One hand only.** Its internal hand detector is hard-coded to `num_hands=1`, so the second hand of every two-handed sign was thrown away. That is why 28 of the 57 TunSL signs could not be trained. Those 28 include the most useful ones for public services: `baladya` (municipality), `demande` (request), `wzara` (ministry), `cv`, `n3awnek` (I help you), `3aslema` (hello), `bousta` (post office), `banka` (bank), and `ma7kma` (court).
2. **One still frame only.** It classifies a single frame, so it has no notion of movement, and the app had to make the user freeze for 700 ms.

**What this pipeline does instead**

| Stage | Where | What |
|---|---|---|
| 1. Landmarks | isolated Python 3.11 venv, MediaPipe Tasks `HandLandmarker` (`num_hands=2`) | Every image or video frame becomes `(2 hands × 21 points × xyz)`. A missing hand is `NaN`. Video is sampled at 15 fps. |
| 2. Features | notebook kernel (NumPy) | A 32-frame window (about 2 s) becomes `32 × 132` features: each hand's shape relative to its wrist, plus each wrist's position relative to the window's own anchor. So the features see **both hands and their motion**, and they don't depend on where the person stands in the frame. |
| 3. Model | notebook kernel (PyTorch, GPU) | A small temporal conv net (about 0.2 M parameters) over position and velocity, with attention pooling. |
| 4. Export | `/kaggle/working/export/` | `sign_model.onnx` for the browser (read and run by the app's own `@souty/sign-model` package: no ONNX runtime to download), `sign_model.meta.json` (labels, window, feature layout, live thresholds), `parity_fixtures.json` (raw landmarks → expected features → expected probabilities, so the TypeScript preprocessing can be unit-tested against this exact Python), and the `hand_landmarker.task` used for extraction. The app must ship the same file. |

**Data sources (all optional except a `none` source)**

* `warcoder/tunisian-sign-language-dataset`: 57 real TunSL signs, 4423 images, 7 signers, CC BY 4.0. **All 57 are now usable**, two-handed ones included. These are still images, so the model learns them as *held* signs.
* `innominate817/hagrid-classification-512p-no-gesture`: the 10 universal gestures already in the app, plus `no_gesture`, which becomes the required `none` class.
* **Your own recorded clips** (the only route to real motion, see the last section): `<label>/<signer>__<anything>.mp4`. This can include new labels and more examples of existing ones.

**Setup:** Settings → Accelerator → **GPU**. Settings → **Internet on** (needed for pip and the hand-landmark model). Add Data → attach the two datasets above (and your clips dataset if you have one). Then Run All.

## 1. Install

MediaPipe pins its own NumPy and protobuf, and Kaggle's kernel Python version moves over time. So landmark extraction runs in its **own `uv`-managed Python 3.11 venv**, called as a subprocess, and never touches the kernel's packages. Training uses the kernel's preinstalled PyTorch.

In [ ]:
!pip install -q uv
!uv venv -q --python 3.11 /kaggle/tmp/mpenv
!uv pip install -q --python /kaggle/tmp/mpenv/bin/python "mediapipe==0.10.21"
!pip install -q onnx onnxruntime
!mkdir -p /kaggle/working/assets
!wget -q -O /kaggle/working/assets/hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task
!MPLBACKEND=Agg /kaggle/tmp/mpenv/bin/python -c "import mediapipe, cv2, numpy; print('mediapipe', mediapipe.__version__, '| opencv', cv2.__version__, '| numpy', numpy.__version__)"
!ls -la /kaggle/working/assets

## 2. Configuration

Dataset roots are **found by slug** under `/kaggle/input`, so it doesn't matter whether Kaggle mounts them at `/kaggle/input/<slug>` or `/kaggle/input/datasets/<owner>/<slug>`. A dataset that isn't attached is skipped with a message, not an error.

`RECORDED_SLUG` is the slug of your own clips dataset (leave it as is if you have none yet). `MIN_SAMPLES` drops any class with too few usable examples after hand detection. Dropped classes are printed in step 4.

In [ ]:
import json
import os
from pathlib import Path

WORK = Path(os.environ.get("SOUTY_WORK", "/kaggle/working"))
INPUT = Path(os.environ.get("SOUTY_INPUT", "/kaggle/input"))
MP_PYTHON = os.environ.get("SOUTY_MP_PYTHON", "/kaggle/tmp/mpenv/bin/python")

TUNSL_SLUG = "tunisian-sign-language-dataset"
HAGRID_SLUG = "hagrid-classification-512p-no-gesture"
RECORDED_SLUG = "souty-sign-clips"  # your own clips dataset, if attached

# Model contract, shared with the app (written to sign_model.meta.json).
WINDOW_FRAMES = 32  # about 2.1 s at 15 fps
TARGET_FPS = 15
MIN_SAMPLES = 15  # drop classes with fewer usable examples than this

# HaGRID folder -> app label (unchanged from the previous model, so the
# vocabulary package keeps working). "no_gesture" is the negative class.
HAGRID_LABEL_MAP = {
    "palm": "Open_Palm",
    "fist": "Closed_Fist",
    "like": "Thumb_Up",
    "dislike": "Thumb_Down",
    "peace": "Victory",
    "one": "Pointing_Up",
    "ok": "OK_Sign",
    "three": "Three_Fingers",
    "four": "Four_Fingers",
    "call": "Shaka_Sign",
    "no_gesture": "none",
}
HAGRID_CAP = 250  # images per HaGRID gesture class (it has 10k+)
NONE_CAP = 600  # "none" covers far more variety, so it gets more

# TunSL folders to leave out on purpose (none by default: two-handed signs
# are learnable now). Labels are ASCII-folded folder names.
TUNSL_EXCLUDE = set()


def find_input(slug):
    """First directory named `slug` under INPUT (depth <= 4), or None."""
    if not INPUT.exists():
        return None
    for dirpath, dirnames, _ in os.walk(INPUT):
        if Path(dirpath).name == slug:
            return Path(dirpath)
        if len(Path(dirpath).relative_to(INPUT).parts) >= 4:
            dirnames.clear()
    return None


roots = {
    "tunsl": find_input(TUNSL_SLUG),
    "hagrid": find_input(HAGRID_SLUG),
    "recorded": find_input(RECORDED_SLUG),
}
for name, root in roots.items():
    print(f"{name:9s} -> {root if root else 'not attached, skipped'}")

CONFIG = {
    "hand_model": str(WORK / "assets" / "hand_landmarker.task"),
    "out": str(WORK / "landmarks.npz"),
    "stats_out": str(WORK / "landmarks_stats.json"),
    "target_fps": TARGET_FPS,
    "min_detection_confidence": 0.4,
    "sources": [
        {"name": "tunsl", "root": str(roots["tunsl"]) if roots["tunsl"] else None,
         "kind": "leaf_folders", "exclude": sorted(TUNSL_EXCLUDE)},
        {"name": "hagrid", "root": str(roots["hagrid"]) if roots["hagrid"] else None,
         "kind": "mapped_folders", "map": HAGRID_LABEL_MAP,
         "cap": HAGRID_CAP, "caps": {"none": NONE_CAP}},
        {"name": "recorded", "root": str(roots["recorded"]) if roots["recorded"] else None,
         "kind": "leaf_folders", "exclude": []},
    ],
}
CONFIG_PATH = WORK / "extract_config.json"
WORK.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))
assert Path(CONFIG["hand_model"]).exists(), "hand_landmarker.task missing: re-run step 1"
assert roots["hagrid"] or roots["recorded"], "need a 'none' source: attach HaGRID or record a none/ folder"

## 3. Extract hand landmarks (both hands, every frame)

This script runs inside the MediaPipe venv. Per sample it stores a `(frames, 2, 21, 3)` array:

* **Coordinates:** `x` and `z` are multiplied by the image aspect ratio (`width / height`). That puts all three axes in *image-height units*, so shapes don't stretch between 224×224 dataset crops and a 640×480 phone camera. The app must do the same.
* **Hand slots:** slot 0 is the hand MediaPipe labels `"Left"`, slot 1 is `"Right"`. If both get the same label (it happens), the hand with the smaller wrist `x` goes to slot 0. The app must use the same rule. Mirror augmentation in training makes the model robust to left-handed signers and front/back camera flips.
* **Images** use IMAGE mode, one frame each. **Videos** use VIDEO mode (tracking), resampled to 15 fps, with leading and trailing hand-less frames trimmed.
* Images and clips where no hand is found are dropped and counted per class.

In [ ]:
%%writefile /kaggle/working/extract_landmarks.py
"""Extracts two-hand MediaPipe landmarks from images and video clips.

Usage: python extract_landmarks.py <extract_config.json>
Runs in the dedicated MediaPipe venv (see the notebook's step 1).
"""
import json
import os
import sys
import time
import unicodedata
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks.python import BaseOptions, vision

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VIDEO_EXTS = {".mp4", ".webm", ".mov", ".avi", ".mkv", ".m4v"}
MEDIA_EXTS = IMAGE_EXTS | VIDEO_EXTS
MAX_EDGE_EMPTY = 2  # hand-less frames kept at each end of a clip


def sanitize_label(name):
    """ASCII-folds accents (e.g. metro's e-acute) so labels stay plain ASCII."""
    folded = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    return folded.strip() or name


def leaf_class_dirs(root):
    """Directories holding media files and no subdirectories: one class each."""
    leaves = []
    for dirpath, dirnames, filenames in os.walk(root):
        if not dirnames and any(Path(f).suffix.lower() in MEDIA_EXTS for f in filenames):
            leaves.append(Path(dirpath))
    return sorted(leaves)


def media_files(d):
    """Sorted media files of a leaf class folder. Names only: no per-file
    stat, which costs ~2 s per kept image on Kaggle's input mount when a
    HaGRID folder holds ~30k files and only a capped few are used."""
    with os.scandir(d) as entries:
        names = sorted(e.name for e in entries if Path(e.name).suffix.lower() in MEDIA_EXTS)
    return [d / n for n in names]


def assign_slots(hands):
    """hands: [(handedness, wrist_x, (21,3) array)] -> (2,21,3), NaN = missing.

    Slot 0 = MediaPipe "Left", slot 1 = "Right"; on a label clash the hand
    with the smaller wrist x takes slot 0. The app must use the same rule.
    """
    out = np.full((2, 21, 3), np.nan, np.float32)
    if len(hands) == 1:
        label, _, arr = hands[0]
        out[0 if label == "Left" else 1] = arr
    elif len(hands) >= 2:
        a, b = hands[0], hands[1]
        if a[0] != b[0]:
            for label, _, arr in (a, b):
                out[0 if label == "Left" else 1] = arr
        else:
            a, b = sorted((a, b), key=lambda h: h[1])
            out[0], out[1] = a[2], b[2]
    return out


def frame_from_result(result, aspect):
    hands = []
    for lms, handedness in zip(result.hand_landmarks, result.handedness):
        arr = np.array([[p.x * aspect, p.y, p.z * aspect] for p in lms], np.float32)
        hands.append((handedness[0].category_name, float(arr[0, 0]), arr))
    return assign_slots(hands), len(hands)


def make_landmarker(cfg, mode):
    options = vision.HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=cfg["hand_model"]),
        running_mode=mode,
        num_hands=2,
        min_hand_detection_confidence=cfg["min_detection_confidence"],
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return vision.HandLandmarker.create_from_options(options)


def to_mp_image(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))


def extract_image(landmarker, path):
    bgr = cv2.imread(str(path))
    if bgr is None:
        return None, 0
    h, w = bgr.shape[:2]
    frame, n = frame_from_result(landmarker.detect(to_mp_image(bgr)), w / h)
    return (frame[None], n) if n else (None, 0)


def extract_video(cfg, path):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    if not 1 <= fps <= 240:
        fps = 30.0
    step_ms = 1000.0 / cfg["target_fps"]
    frames, counts = [], []
    next_ms, idx = 0.0, 0
    with make_landmarker(cfg, vision.RunningMode.VIDEO) as landmarker:
        while True:
            ok, bgr = cap.read()
            if not ok:
                break
            t_ms = idx * 1000.0 / fps
            idx += 1
            if t_ms + 1e-6 < next_ms:
                continue
            next_ms += step_ms
            h, w = bgr.shape[:2]
            result = landmarker.detect_for_video(to_mp_image(bgr), int(t_ms))
            frame, n = frame_from_result(result, w / h)
            frames.append(frame)
            counts.append(n)
    cap.release()
    present = [i for i, n in enumerate(counts) if n]
    if not present:
        return None, 0
    lo = max(0, present[0] - MAX_EDGE_EMPTY)
    hi = min(len(frames), present[-1] + 1 + MAX_EDGE_EMPTY)
    return np.stack(frames[lo:hi]), max(counts)


def classes_for(src):
    root = Path(src["root"])
    if src["kind"] == "leaf_folders":
        excluded = set(src.get("exclude", []))
        for d in leaf_class_dirs(root):
            label = sanitize_label(d.name)
            if label not in excluded:
                yield label, d, sanitize_label(d.parent.name), None
    elif src["kind"] == "mapped_folders":
        by_name = {d.name: d for d in leaf_class_dirs(root)}
        for folder, label in src["map"].items():
            if folder not in by_name:
                print(f"  WARNING [{src['name']}] no '{folder}' folder, skipping {label}")
                continue
            cap = src.get("caps", {}).get(label, src.get("cap"))
            yield label, by_name[folder], src["name"], cap


def main():
    cfg = json.loads(Path(sys.argv[1]).read_text())
    all_frames, offsets = [], [0]
    meta = {k: [] for k in ("label", "source", "category", "group", "order", "n_hands", "kind", "path")}
    stats = []
    t0 = time.time()

    with make_landmarker(cfg, vision.RunningMode.IMAGE) as image_landmarker:
        for src in cfg["sources"]:
            if not src.get("root") or not Path(src["root"]).exists():
                print(f"[{src['name']}] not attached, skipped")
                continue
            print(f"[{src['name']}] {src['root']}")
            for label, d, category, cap in classes_for(src):
                files = media_files(d)[:cap] if cap else media_files(d)
                kept = 0
                for order, f in enumerate(files):
                    if f.suffix.lower() in VIDEO_EXTS:
                        seq, n_hands = extract_video(cfg, f)
                        kind = "video"
                    else:
                        seq, n_hands = extract_image(image_landmarker, f)
                        kind = "image"
                    if seq is None:
                        continue
                    kept += 1
                    all_frames.append(seq)
                    offsets.append(offsets[-1] + len(seq))
                    meta["label"].append(label)
                    meta["source"].append(src["name"])
                    meta["category"].append(category)
                    meta["group"].append(f.stem.split("__")[0] if "__" in f.stem else "")
                    meta["order"].append(order)
                    meta["n_hands"].append(n_hands)
                    meta["kind"].append(kind)
                    meta["path"].append(str(f.relative_to(src["root"])))
                stats.append({"source": src["name"], "label": label, "category": category,
                              "files": len(files), "kept": kept})
                print(f"  {label:16s} kept {kept:4d}/{len(files):4d}  ({time.time() - t0:5.0f}s)")

    if not all_frames:
        raise SystemExit("No samples extracted: check the dataset roots in step 2.")
    np.savez_compressed(
        cfg["out"],
        frames=np.concatenate(all_frames).astype(np.float32),
        offsets=np.asarray(offsets, np.int64),
        **{k: np.asarray(v) for k, v in meta.items()},
    )
    Path(cfg["stats_out"]).write_text(json.dumps(stats, indent=1))
    print(f"Saved {len(offsets) - 1} samples, {offsets[-1]} frames -> {cfg['out']}")


if __name__ == "__main__":
    main()

In [ ]:
import subprocess
import sys


def run_streaming(cmd):
    """Runs a command and streams its output into this cell."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, "MPLBACKEND": "Agg", "GLOG_minloglevel": "2"})
    for line in proc.stdout:
        if "inference_feedback_manager" in line or line.startswith(("W0000", "I0000")):
            continue  # MediaPipe graph chatter
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")


run_streaming([MP_PYTHON, str(WORK / "extract_landmarks.py"), str(CONFIG_PATH)])

## 4. Audit the extracted data

**Read this before trusting a training run.** For each class it shows usable samples, the share of samples where **both** hands were found, and what was dropped. Two things to look for:

* A sign you know is two-handed but shows a low `two_hands` share. Hand detection is missing a hand there. Lower `min_detection_confidence` in step 2, or record clips.
* Classes listed under *dropped (< MIN_SAMPLES)*. They aren't trained. Record clips for them if you need them.

In [ ]:
import numpy as np
import pandas as pd

data = np.load(CONFIG["out"])
frames_all, offsets = data["frames"], data["offsets"]
meta = pd.DataFrame({k: data[k] for k in ("label", "source", "category", "group", "order", "n_hands", "kind", "path")})
meta["n_frames"] = np.diff(offsets)
sequences = [frames_all[offsets[i]:offsets[i + 1]] for i in range(len(meta))]

stats = pd.DataFrame(json.loads(Path(CONFIG["stats_out"]).read_text()))
audit = (meta.groupby(["source", "label"])
         .agg(samples=("label", "size"), two_hands=("n_hands", lambda s: (s == 2).mean()),
              videos=("kind", lambda s: (s == "video").sum()), signers=("group", lambda s: s[s != ""].nunique()))
         .reset_index()
         .merge(stats[["source", "category", "label", "files"]], on=["source", "label"], how="right")
         .fillna({"samples": 0, "two_hands": 0.0, "videos": 0, "signers": 0}))
audit["dropped_no_hand"] = audit["files"] - audit["samples"]

counts = meta.groupby("label").size()
too_thin = sorted(counts[counts < MIN_SAMPLES].index)
if "none" in too_thin:
    raise SystemExit("'none' is below MIN_SAMPLES: attach HaGRID or record more none/ clips")
keep = np.flatnonzero(~meta["label"].isin(too_thin).to_numpy())
meta = meta.iloc[keep].reset_index(drop=True)
sequences = [sequences[i] for i in keep]

LABELS = sorted(meta["label"].unique(), key=lambda l: (l != "none", l))  # "none" is class 0
LABEL_TO_ID = {l: i for i, l in enumerate(LABELS)}
y_all = meta["label"].map(LABEL_TO_ID).to_numpy()

with pd.option_context("display.max_rows", 200, "display.width", 160):
    print(audit.sort_values(["source", "category", "label"]).to_string(
        index=False, formatters={"two_hands": "{:.0%}".format}))
print(f"\nDropped (< MIN_SAMPLES={MIN_SAMPLES}): {too_thin or 'none'}")
print(f"Training on {len(LABELS)} classes, {len(meta)} samples "
      f"({(meta['kind'] == 'video').sum()} video clips, {(meta['kind'] == 'image').sum()} images)")

## 5. Train / validation / test split

* **Recorded clips with ≥ 3 signers:** split **by signer**. The last signer (sorted) is test and the second-to-last is validation. This is the honest "does it work on a stranger" number. Name your files `<signer>__<anything>.mp4`.
* **Everything else:** contiguous blocks of each class's capture-ordered files (80 / 10 / 10). The TunSL photos were taken in sequence with laptop cameras, so neighbouring files are near-duplicates (measured with perceptual hashing in the previous pipeline). A random split would leak them across sets and inflate the score.

In [ ]:
VAL_FRACTION = TEST_FRACTION = 0.1

split = np.empty(len(meta), dtype=object)
for (source, label), rows in meta.groupby(["source", "label"]):
    signers = sorted(set(rows["group"]) - {""})
    if source == "recorded" and len(signers) >= 3:
        split[rows.index] = np.where(rows["group"] == signers[-1], "test",
                                     np.where(rows["group"] == signers[-2], "val", "train"))
        continue
    ordered = rows.sort_values("order").index.to_numpy()
    n = len(ordered)
    n_test = max(1, round(n * TEST_FRACTION)) if n >= 3 else 0
    n_val = max(1, round(n * VAL_FRACTION)) if n >= 3 else 0
    split[ordered[: n - n_val - n_test]] = "train"
    split[ordered[n - n_val - n_test: n - n_test]] = "val"
    split[ordered[n - n_test:]] = "test"
meta["split"] = split
print(meta.groupby("split").size().to_string())

## 6. Features (the contract the app must reproduce exactly)

`window_features(win)` maps one window `(T=32, 2, 21, 3)` of raw landmarks (NaN = missing hand) to `(32, 132)`. Per hand slot, 66 values:

| index | value |
|---|---|
| 0–62 | 21 landmarks × (x, y, z) **minus the wrist**, divided by **that frame's palm size** (the 2D distance from the wrist, landmark 0, to the middle-finger base, landmark 9). This is the hand shape, independent of position and size. |
| 63–64 | wrist (x, y) **minus the window anchor** (the mean of every present wrist in the window), divided by the **window scale** (the median palm size in the window). This is where each hand is and how it moves, relative to the other hand and to its own path. It is the same wherever the user stands and however far they are from the camera. |
| 65 | 1 if the hand is present in this frame, else 0 (and all 66 values are 0) |

Slot 0 comes first, then slot 1. All values are clipped to [-10, 10]. Velocity is computed **inside the model** (frame-to-frame difference), so the app only reproduces this function. The exported `parity_fixtures.json` contains raw windows with the expected output of this function and of the model, for a unit test on the TypeScript side.

**Windows:** clips are fitted to 32 frames by random crop (training) or centre crop (eval), or padded by repeating the edge frames. Eval windows are right-aligned, which matches the live app: the window ends as the sign ends. A still image becomes 32 copies of itself plus per-frame jitter, which matches a user holding a sign. **Augmentation:** mirror (swap slots and negate x), rotation ±15°, anisotropic scale ±10%, speed ×0.75–1.33, landmark jitter, per-frame hand dropout, and hands not yet raised at the start of the window.

In [ ]:
N_FEATURES = 132


def window_features(win):
    """(T, 2, 21, 3) raw landmarks, NaN = missing hand -> (T, 132) float32."""
    win = np.asarray(win, np.float32)
    t = win.shape[0]
    present = ~np.isnan(win[:, :, 0, 0])  # (T, 2)
    feats = np.zeros((t, 2, 66), np.float32)
    if not present.any():
        return feats.reshape(t, N_FEATURES)
    safe = np.where(present[:, :, None, None], win, 0.0)
    wrist = safe[:, :, 0, :]  # (T, 2, 3)
    palm = np.linalg.norm(safe[:, :, 9, :2] - safe[:, :, 0, :2], axis=-1)  # (T, 2)
    palm = np.maximum(palm, 1e-3)
    scale = max(float(np.median(palm[present])), 1e-3)
    anchor = wrist[present][:, :2].mean(axis=0)  # (2,)
    feats[:, :, :63] = ((safe - wrist[:, :, None, :]) / palm[:, :, None, None]).reshape(t, 2, 63)
    feats[:, :, 63:65] = (wrist[:, :, :2] - anchor) / scale
    feats[:, :, 65] = 1.0
    feats[~present] = 0.0
    return np.clip(feats, -10.0, 10.0).reshape(t, N_FEATURES)


def resample(seq, speed):
    n = len(seq)
    if n == 1:
        return seq
    m = max(1, int(round(n / speed)))
    return seq[np.round(np.linspace(0, n - 1, m)).astype(int)]


def fit_window(seq, train, rng, t=WINDOW_FRAMES):
    n = len(seq)
    if n >= t:
        start = int(rng.integers(0, n - t + 1)) if train else (n - t) // 2
        return seq[start:start + t].copy()
    pad = t - n
    left = int(rng.integers(0, pad + 1)) if train else pad
    out = np.concatenate([np.repeat(seq[:1], left, 0), seq, np.repeat(seq[-1:], pad - left, 0)])
    if train and left > 0 and rng.random() < 0.3:
        out[: int(rng.integers(1, left + 1))] = np.nan  # hands not raised yet
    return out


def augment(seq, rng):
    seq = seq.copy()
    if rng.random() < 0.5:  # mirror: left-handed signer / other camera
        seq = seq[:, ::-1].copy()
        seq[..., 0] *= -1
    a = np.deg2rad(rng.uniform(-15, 15))
    c, s = np.cos(a), np.sin(a)
    x, y = seq[..., 0].copy(), seq[..., 1].copy()
    seq[..., 0] = (c * x - s * y) * rng.uniform(0.9, 1.1)
    seq[..., 1] = (s * x + c * y) * rng.uniform(0.9, 1.1)
    return resample(seq, rng.uniform(0.75, 1.33))


def jitter(win, rng):
    win = win + rng.normal(0, 0.004, win.shape).astype(np.float32)
    drop = rng.random(win.shape[:2]) < 0.03
    win[drop] = np.nan
    return win


def make_window(seq, train, rng):
    if train:
        return jitter(fit_window(augment(seq, rng), True, rng), rng)
    return fit_window(seq, False, rng)


# Sanity checks on the contract.
_rng = np.random.default_rng(0)
_s = sequences[0]
_f = window_features(make_window(_s, False, _rng))
assert _f.shape == (WINDOW_FRAMES, N_FEATURES) and np.isfinite(_f).all()
assert np.allclose(window_features(fit_window(_s, False, _rng) + 5.0), _f, atol=1e-4), "must be translation invariant"
print("feature contract OK:", _f.shape)

## 7. Model

The model takes `x` (32 × 132) and its frame-to-frame velocity, applies a linear projection, then 3 residual depthwise-separable temporal convolutions (kernel 5, about 1 s receptive field with the stack), then attention pooling plus max pooling, then logits. About 0.2 M parameters, under 1 MB, and it runs in a few milliseconds in the browser. It uses only plain ops (Linear, Conv1d, BatchNorm, GELU, softmax). The app's `packages/sign-model` reads the weights straight out of the ONNX file and runs this exact forward pass in TypeScript, so **changing this architecture means changing `packages/sign-model/src/signnet.ts` too** (its ONNX reader rejects a graph it does not recognise, and its parity test checks the probabilities).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


class ConvBlock(nn.Module):
    def __init__(self, width, kernel, dropout):
        super().__init__()
        self.dw = nn.Conv1d(width, width, kernel, padding=kernel // 2, groups=width)
        self.bn = nn.BatchNorm1d(width)
        self.pw = nn.Conv1d(width, width, 1)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return x + self.drop(F.gelu(self.pw(self.bn(self.dw(x)))))


class SignNet(nn.Module):
    def __init__(self, n_features, n_classes, width=192, depth=3, kernel=5, dropout=0.25):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(2 * n_features, width), nn.LayerNorm(width), nn.GELU())
        self.blocks = nn.Sequential(*[ConvBlock(width, kernel, dropout) for _ in range(depth)])
        self.attn = nn.Linear(width, 1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * width, n_classes))

    def forward(self, x):  # (B, T, F)
        dx = torch.cat([torch.zeros_like(x[:, :1]), x[:, 1:] - x[:, :-1]], dim=1)
        h = self.inp(torch.cat([x, dx], dim=-1))
        h = self.blocks(h.transpose(1, 2)).transpose(1, 2)  # (B, T, W)
        w = torch.softmax(self.attn(h), dim=1)
        pooled = torch.cat([(w * h).sum(dim=1), h.amax(dim=1)], dim=-1)
        return self.head(pooled)


class WindowDataset(torch.utils.data.Dataset):
    def __init__(self, rows, train):
        self.rows, self.train = np.asarray(rows), train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = int(self.rows[i])
        rng = np.random.default_rng() if self.train else np.random.default_rng(r)
        return torch.from_numpy(window_features(make_window(sequences[r], self.train, rng))), int(y_all[r])


model = SignNet(N_FEATURES, len(LABELS)).to(DEVICE)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M parameters, {len(LABELS)} classes")

## 8. Train

Class-balanced sampling (weights ∝ 1/√count) keeps the big HaGRID and `none` pools from drowning the thin TunSL classes. Training uses AdamW with a one-cycle LR and label smoothing. The **best epoch by validation macro-recall** (mean per-class accuracy, so a few big classes can't hide bad small ones) is kept, with early stopping.

In [ ]:
import copy
import time

EPOCHS = int(os.environ.get("SOUTY_EPOCHS", 80))
BATCH = 128
PATIENCE = 15
WORKERS = 0 if os.name == "nt" else 2

idx = {s: np.flatnonzero(meta["split"].to_numpy() == s) for s in ("train", "val", "test")}
class_counts = np.bincount(y_all[idx["train"]], minlength=len(LABELS)).clip(min=1)
sampler = torch.utils.data.WeightedRandomSampler(
    (1.0 / np.sqrt(class_counts))[y_all[idx["train"]]], num_samples=len(idx["train"]), replacement=True)
loaders = {
    "train": torch.utils.data.DataLoader(WindowDataset(idx["train"], True), BATCH, sampler=sampler,
                                         num_workers=WORKERS, drop_last=len(idx["train"]) > BATCH),
    "val": torch.utils.data.DataLoader(WindowDataset(idx["val"], False), 256, num_workers=WORKERS),
    "test": torch.utils.data.DataLoader(WindowDataset(idx["test"], False), 256, num_workers=WORKERS),
}


@torch.no_grad()
def predict(loader):
    model.eval()
    probs, ys = [], []
    for xb, yb in loader:
        probs.append(torch.softmax(model(xb.to(DEVICE)), -1).cpu())
        ys.append(yb)
    return torch.cat(probs).numpy(), torch.cat(ys).numpy()


def macro_recall(probs, ys):
    pred = probs.argmax(1)
    return float(np.mean([(pred[ys == c] == c).mean() for c in np.unique(ys)]))


opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.05)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=EPOCHS * len(loaders["train"]))
best = (-1.0, None, 0)
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for xb, yb in loaders["train"]:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb, label_smoothing=0.1)
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()
        loss_sum += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    val_probs, val_y = predict(loaders["val"])
    val_mr = macro_recall(val_probs, val_y)
    if val_mr > best[0]:
        best = (val_mr, copy.deepcopy(model.state_dict()), epoch)
    if epoch % 5 == 0 or epoch == 1 or val_mr == best[0]:
        print(f"epoch {epoch:3d}  loss {loss_sum / total:.3f}  train acc {correct / total:.3f}  "
              f"val macro-recall {val_mr:.3f}{'  *' if val_mr == best[0] else ''}  ({time.time() - t0:.0f}s)")
    if epoch - best[2] >= PATIENCE:
        print(f"early stop: no val improvement for {PATIENCE} epochs")
        break
model.load_state_dict(best[1])
print(f"best val macro-recall {best[0]:.3f} at epoch {best[2]}")

## 9. Test: per class, worst first

The aggregate number hides bad classes, so read the per-class table. A class under about 70% here will misfire live: record clips for it or exclude it. The confusion list shows which pairs the model mixes up. Signs that differ **only by movement** will stay confused until real clips of them are recorded, because still images carry no motion.

In [ ]:
test_probs, test_y = predict(loaders["test"])
test_pred = test_probs.argmax(1)
TEST_ACC = float((test_pred == test_y).mean())
TEST_MACRO = macro_recall(test_probs, test_y)
print(f"test accuracy {TEST_ACC:.3f}   test macro-recall {TEST_MACRO:.3f}   ({len(test_y)} windows)\n")

per_class = []
for c in np.unique(test_y):
    m = test_y == c
    wrong = test_pred[m][test_pred[m] != c]
    top_conf = LABELS[np.bincount(wrong).argmax()] if len(wrong) else ""
    per_class.append({"label": LABELS[c], "n_test": int(m.sum()), "recall": float((test_pred[m] == c).mean()),
                      "mostly_confused_with": top_conf})
per_class = pd.DataFrame(per_class).sort_values("recall")
with pd.option_context("display.max_rows", 200):
    print(per_class.to_string(index=False, formatters={"recall": "{:.0%}".format}))

## 10. Export for the app

This step writes `/kaggle/working/export/`. **Download the whole folder** from the Output panel.

* `sign_model.onnx`: input `frames` `(batch, 32, 132)` float32, output `logits` `(batch, n_classes)`. Apply softmax in the app. The export is checked against PyTorch with `onnxruntime` before it is written.
* `sign_model.meta.json`: labels in output order (`none` is index 0), the window and fps, the feature layout, the hand-slot rule, the recommended live decoding thresholds, and test scores.
* `parity_fixtures.json`: 6 raw test windows (`null` = missing hand) with the expected `features` and `probs`. Copy it to `packages/sign-model/test/`: `pnpm --filter @souty/sign-model test` checks that the app reproduces them to about 1e-4.
* `hand_landmarker.task`: the exact landmark model used for extraction. Ship this one in the app.

**Live decoding (recommended defaults in the meta file):** keep a ring buffer of the last 32 frames, sampled at 15 fps. Run the model every 2 frames while at least one hand is visible. Emit a word when the same non-`none` class wins with probability ≥ `min_confidence` on `consecutive_hits` evaluations in a row. Then don't emit that class again until a different class or `none` has won. That replaces the old "freeze for 700 ms" rule.

In [ ]:
import datetime
import shutil

import onnxruntime as ort

EXPORT = WORK / "export"
EXPORT.mkdir(parents=True, exist_ok=True)
model.eval().cpu()

dummy = torch.zeros(1, WINDOW_FRAMES, N_FEATURES)
onnx_path = EXPORT / "sign_model.onnx"
export_kwargs = dict(input_names=["frames"], output_names=["logits"], opset_version=17,
                     dynamic_axes={"frames": {0: "batch"}, "logits": {0: "batch"}})
try:
    torch.onnx.export(model, dummy, str(onnx_path), dynamo=False, **export_kwargs)
except TypeError:  # older torch without the `dynamo` switch
    torch.onnx.export(model, dummy, str(onnx_path), **export_kwargs)

# Verify the ONNX graph against PyTorch on real test windows.
fixture_rows = list(idx["test"][np.linspace(0, len(idx["test"]) - 1, min(6, len(idx["test"]))).astype(int)])
two_handed = [r for r in idx["test"] if meta.loc[r, "n_hands"] == 2]
if two_handed and not any(meta.loc[r, "n_hands"] == 2 for r in fixture_rows):
    fixture_rows[-1] = two_handed[0]
windows = [fit_window(sequences[r], False, np.random.default_rng(int(r))) for r in fixture_rows]
feats = np.stack([window_features(w) for w in windows])
with torch.no_grad():
    torch_probs = torch.softmax(model(torch.from_numpy(feats)), -1).numpy()
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
onnx_probs = torch.softmax(torch.from_numpy(session.run(None, {"frames": feats})[0]), -1).numpy()
max_diff = float(np.abs(onnx_probs - torch_probs).max())
assert max_diff < 1e-4, f"ONNX diverges from PyTorch: {max_diff}"
print(f"ONNX verified (max prob diff {max_diff:.2e}), {onnx_path.stat().st_size / 1024:.0f} KB")


def encode_window(win):
    # Full float32 precision: shape values divide by the palm size, so a
    # small hand magnifies any rounding of the raw landmarks.
    return [[None if np.isnan(hand[0, 0]) else hand.astype(float).tolist() for hand in frame] for frame in win]


(EXPORT / "parity_fixtures.json").write_text(json.dumps({
    "tolerance": 1e-4,
    "cases": [{"label": LABELS[int(y_all[r])], "source_path": str(meta.loc[r, "path"]),
               "frames": encode_window(w), "features": np.round(f, 6).tolist(), "probs": np.round(p, 6).tolist()}
              for r, w, f, p in zip(fixture_rows, windows, feats, onnx_probs)],
}))

model_meta = {
    "format": "souty-sign-seq/1",
    "created": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "labels": LABELS,
    "negative_label": "none",
    "window_frames": WINDOW_FRAMES,
    "fps": TARGET_FPS,
    "num_features": N_FEATURES,
    "landmarks": {
        "model": "hand_landmarker.task (MediaPipe Tasks, float16)",
        "num_hands": 2,
        "coordinates": "x and z multiplied by frame width/height; y unchanged",
        "slots": "slot 0 = handedness 'Left', slot 1 = 'Right'; on a clash the smaller wrist x takes slot 0",
    },
    "features": {
        "per_slot": 66,
        "layout": "[slot0: 63 shape (landmark - wrist) / frame palm, 2 wrist xy (wrist - window anchor) / window scale, "
                  "1 present] [slot1: same]; missing hand = all zeros; clip [-10, 10]",
        "palm": "2D distance landmark 0 -> landmark 9",
        "window_anchor": "mean xy of every present wrist in the window",
        "window_scale": "median palm over every present hand in the window (min 1e-3)",
    },
    "decoding": {"stride_frames": 2, "min_confidence": 0.6, "consecutive_hits": 3},
    "metrics": {"test_accuracy": round(TEST_ACC, 4), "test_macro_recall": round(TEST_MACRO, 4),
                "val_macro_recall": round(best[0], 4),
                "per_class_recall": {r.label: round(r.recall, 4) for r in per_class.itertuples()}},
    "training_data": sorted(meta["source"].unique().tolist()),
}
(EXPORT / "sign_model.meta.json").write_text(json.dumps(model_meta, indent=2, ensure_ascii=False))
shutil.copy(CONFIG["hand_model"], EXPORT / "hand_landmarker.task")
for f in sorted(EXPORT.iterdir()):
    print(f"  {f.name:24s} {f.stat().st_size / 1024:8.0f} KB")

## 11. Getting real motion and the vocabulary you actually need

**What the public data can and cannot do.** No free dataset of *moving* Tunisian signs exists today. The TunSL dataset above is still photos, taken with laptop cameras (its README says so); no video of it exists publicly. The few other Tunisian resources are not usable for training: a 279-sign online dictionary and a small narrative video corpus both forbid copying or derivatives (NC-ND licences), and a student medical-vocabulary video project has not released its videos. Kaggle does have large **video and landmark** sign datasets, but in *other* sign languages: Google's ISLR competition (`asl-signs`, 250 ASL signs, about 94k landmark sequences), KArSL-502 (Saudi/unified Arabic SL), and Isharah-1000 (Saudi SL). Training Souty on them would teach users signs a Tunisian Deaf person doesn't use. At most they could **pre-train** this same network before fine-tuning on Tunisian data. That is a later optimisation, not a substitute.

So the path to real movement-based TunSL signs is **recording clips**, ideally with ATILS interpreters and Deaf signers. This notebook already accepts them, so no code changes are needed:

```
souty-sign-clips/                 ← private Kaggle dataset (slug = RECORDED_SLUG)
  baladya/   amira__01.mp4  amira__02.mp4  sami__01.mp4 ...
  mada9_wled/ ...                 ← new label: just a new folder
  none/      amira__idle.mp4  sami__scratch_head.mp4 ...
```

* **Per sign:** at least 5 different signers × 6 repetitions. Clips of 1.5–3 s, each containing **one** sign, starting and ending with the hands low.
* **Camera:** phone at chest height, about 1 m away, upper body in frame (head can be cut), 15–30 fps. Vary the room, light, and clothing.
* **`none/` matters as much as the signs.** Record idle hands, scratching, waving at someone, phone use, and transitions between signs. It is what stops false words in continuous use.
* Filenames must start with `<signer>__`. With 3 or more signers, the test score becomes a held-out-person score.
* A label that exists in both TunSL images and recorded clips is merged into one class, so the clips add motion to an existing sign.

**Vocabulary already trainable from TunSL (now that two-handed signs work), for public services and daily life:** `baladya` (municipality), `demande` (request/application), `wzara` (ministry), `ma7kma` (court), `bousta` (post office), `banka` (bank), `cv`, `n3awnek` (I help you), `nekteblk` (I write for you), `3aslema` (hello), `mar7ba` (welcome), `se7a` (health / thank you), `t7eb` (do you want), `enti` (you), `dar` (house), `3ayla` (family), `om` (mother), `o5t` (sister), `5ou` (brother), the days of the week, and transport.

**Suggested recordings for municipality counters**, to agree with ATILS on the correct TunSL sign for each before recording: birth certificate (مضمون ولادة), ID card (بطاقة تعريف), signature certification (تعريف بالإمضاء), certified copy (نسخة مطابقة للأصل), residence certificate (شهادة إقامة), marriage certificate (عقد زواج), building permit (رخصة بناء), stamp/fee (طابع), queue number / wait (نوبة), where (وين), how much (قداش), when (وقتاش), today/tomorrow (اليوم / غدوة), thank you (يعيشك), please (عيشك), I don't understand (ما فهمتش), repeat (عاود), write it (اكتبهولي), and paper/document (ورقة).